# 01 · Interpolate EDIs onto common periods

GoFEM data files work best when every station has the same periods. This notebook interpolates
each station's impedance and tipper onto one shared, log-spaced set of periods and writes new EDI
files. The originals are never changed.

* **Needs:** a folder of EDI files (or a `.lst` file listing them).
* **Writes:** `work/01_interpolated_edis/*.edi`, same file names as the originals.
* **Takes:** a few seconds per hundred stations.
* **GUI:** setup tab 0, *Interpolate EDIs*.

A station is never extrapolated: it only gets the periods inside its own range, and the notebook
lists the stations that end up with fewer periods. The sample stations already share one period
set, so here this step mostly just resamples them.

## Settings

Change these for your data. Paths are relative to this notebook's folder (or give full paths).

In [ ]:
EDI_DIR = "sample_data/edis"            # folder of EDIs, or a .lst file listing them
OUT_DIR = "work/01_interpolated_edis"   # new EDIs go here

PERIOD_MIN_S = 0.01         # shortest period wanted (seconds)
PERIOD_MAX_S = 1000.0       # longest period wanted (seconds)
PERIODS_PER_DECADE = 6      # how many periods per factor of 10 in period
INTERP_TYPE = "slinear"     # "slinear" (linear in log-period) or "cubic"

In [ ]:
%matplotlib inline
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np

from mtinv_gui.backends import edi_interp, edi_io

# mtpy warns about dividing by zero for components that are zero (e.g. ZXX of a 1-D site); harmless
warnings.filterwarnings("ignore", category=RuntimeWarning, module="mtpy")

## 1. What periods do the stations have?

Each bar is one station's period range; the dashed lines are the range asked for above. Stations
whose bar stops inside the dashed lines will get fewer periods.

In [ ]:
stations = edi_io.read_edi_headers(EDI_DIR)
print(f"{len(stations)} stations in {EDI_DIR}")

fig, ax = plt.subplots(figsize=(8, max(3, 0.18 * len(stations))))
for k, st in enumerate(stations):
    ax.plot([st.period_min_s, st.period_max_s], [k, k], lw=4, color="tab:blue")
ax.axvline(PERIOD_MIN_S, ls="--", color="k")
ax.axvline(PERIOD_MAX_S, ls="--", color="k")
ax.set_xscale("log")
ax.set_yticks(range(len(stations)))
ax.set_yticklabels([st.station_id for st in stations], fontsize=7)
ax.set_xlabel("period (s)")
ax.set_title("period range of each station")
plt.show()

## 2. The new periods

In [ ]:
periods = edi_interp.periods_from_decade(PERIOD_MIN_S, PERIOD_MAX_S, PERIODS_PER_DECADE)
print(f"{len(periods)} periods from {periods[0]:.4g} s to {periods[-1]:.4g} s:")
print(", ".join(f"{p:.4g}" for p in periods))

## 3. Interpolate and write

Messages below name any station that was skipped, or that got fewer periods than the rest.

In [ ]:
result = edi_interp.interpolate_collection(
    EDI_DIR, OUT_DIR,
    period_min_s=PERIOD_MIN_S, period_max_s=PERIOD_MAX_S,
    periods_per_decade=PERIODS_PER_DECADE, interp_type=INTERP_TYPE)
print(f"\n{len(result['written'])} written, {len(result['skipped'])} skipped -> {OUT_DIR}")

## 4. Check one station: before and after

Points are the original data, lines the interpolated values. They should sit on top of each
other. Change `CHECK_STATION` to look at another one.

In [ ]:
CHECK_STATION = stations[0].station_id

before = edi_interp.read_station_response(
    next(st.path for st in stations if st.station_id == CHECK_STATION))
after = edi_interp.read_station_response(os.path.join(OUT_DIR, CHECK_STATION + ".edi"))

fig, (ax_rho, ax_phi) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
for comp, color in (("xy", "tab:red"), ("yx", "tab:blue")):
    ax_rho.loglog(before["periods"], before["res_" + comp], "o", ms=3, color=color, alpha=0.5)
    ax_rho.loglog(after["periods"], after["res_" + comp], "-", color=color, label=comp)
    ax_phi.semilogx(before["periods"], before["phase_" + comp], "o", ms=3, color=color, alpha=0.5)
    ax_phi.semilogx(after["periods"], after["phase_" + comp], "-", color=color)
ax_rho.set_ylabel("apparent resistivity (ohm-m)")
ax_phi.set_ylabel("phase (degrees)")
ax_phi.set_xlabel("period (s)")
ax_rho.legend()
ax_rho.set_title(f"{CHECK_STATION}: original (points) and interpolated (lines)")
plt.show()

## Next

Use `work/01_interpolated_edis` as the EDI folder in the next notebooks, or in `gateway_mt` to
select the data.

**For coders:** `edi_interp.interpolate_collection` reads each file with mtpy, clips the target
periods to the station's own range and calls `MT.interpolate` / `MT.write_mt_file`. To change
how a station is interpolated (for example, smoothing first), copy that loop from
`mtinv_gui/backends/edi_interp.py` into a cell here and edit it.